---
# **Benchmark NBody Google Colab Free**
---

Estudante: André Destefani Stefanato

Orientador: Prof. Roberto Colistete Júnior

---
## **Bibliotecas importadas**
---

In [ ]:
import numpy as np
import cupy as cp
import platform
import math

---
## **Informações sobre o hardware, o python e as bibliotecas usadas**
---

Nome do sistema operacional, nome do computador na rede, versão do kernel, arquitetura de hardware (32/64 bits), etc:

In [ ]:
!uname -a

Linux Fisica1 6.12.35+1-amd64 #1 SMP PREEMPT_DYNAMIC Debian 6.12.35-1~mx23ahs (2025-07-04) x86_64 GNU/Linux


Nome e versão do sistema operacional:

In [ ]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Debian
Description:	Debian GNU/Linux 12 (bookworm)
Release:	12
Codename:	bookworm


Diversas informações da CPU, como nome do processador, frequência em MHz da CPU, número de núcleos/cores, número de threads, memória cache, arquitetura de hardware (32/64 bits), etc:

In [ ]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             36 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      4
  On-line CPU(s) list:       0-3
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Core(TM) i5-3470 CPU @ 3.20GHz
    CPU family:              6
    Model:                   58
    Thread(s) per core:      1
    Core(s) per socket:      4
    Socket(s):               1
    Stepping:                9
    CPU(s) scaling MHz:      94%
    CPU max MHz:             3600,0000
    CPU min MHz:             1600,0000
    BogoMIPS:                6385,23
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush dts acpi mmx fxsr sse 
                             sse2 ss ht tm pbe syscall nx rdtscp lm constant_tsc
                              arch_perfmon pebs bts rep_good 

Partições do sistema de arquivos do sistema operacional:

In [ ]:
!df -h

Sist. Arq.      Tam. Usado Disp. Uso% Montado em
udev             16G     0   16G   0% /dev
tmpfs           3,2G  2,6M  3,2G   1% /run
/dev/sda1       440G  164G  253G  40% /
tmpfs           5,0M   12K  5,0M   1% /run/lock
tmpfs            15G  8,0K   15G   1% /dev/shm
/dev/sda2       401G   28G  353G   8% /home
cgroup           12K     0   12K   0% /sys/fs/cgroup
tmpfs           3,2G     0  3,2G   0% /run/user/111
tmpfs           3,2G   64K  3,2G   1% /run/user/1002


Memória RAM total e em uso pelo sistema operacional:

In [ ]:
!free

               total       usada       livre    compart.  buff/cache  disponível
Mem.:       32827872     2792868    27876420       25788     2593340    30035004
Swap:       40957948           0    40957948


Versão do compilador C/C++ gcc:

In [ ]:
!gcc --version

gcc (Debian 12.2.0-14+deb12u1) 12.2.0
Copyright (C) 2022 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



Número da versão de Python:

In [ ]:
platform.python_version()

'3.11.2'

data da versão:

In [ ]:
platform.python_build()

('main', 'May 12 2026 05:17:27')

compilador C/C++ utilizado para criar tal versão de Python:

In [ ]:
platform.python_compiler()

'GCC 12.2.0'

Versão de NumPy, onde 'np' é um apelido comum:

In [ ]:
np.__version__

'2.4.6'

Versão de CuPy, onde 'cp' é um apelido comum:

In [ ]:
cp.__version__

'14.1.1'

---
## **Inicialização do problema e fórmulas usadas**
---

Variáveis usadas por todos os códigos

In [ ]:
espacamento = 10
passos = 10
dt = 0.1
G = 1
epsilon = 5
numCorpos = 2**13
corposPorEixo3D = int(math.cbrt(numCorpos))

resto = numCorpos - corposPorEixo3D**3

Fórmulas da força gravitacional, energia cinética e energia potencial gravitacional respectivamente

$$ F = \sum_{\substack{j = 1 \\ j \ne i}}^{N} \frac{G m_i m_j}{|\vec{r}_{ij}|^2} $$

$$ K = \sum^N_{i=1} \frac{m_i |\vec{v}_i|^2}{2} $$

$$ U = \sum^N_{i=1}\sum_{\substack{j = 1 \\ j \ne i}}^{N} \frac{G m_i m_j}{|r_{ij}|} $$

---
## **Numpy**
---

### <font color="red">3D</font>

Inicialização dos atributos feita em Python puro, essa parte do código sera executada apenas uma vez. Os vetores gerados serão jogados para o Numpy depois

In [ ]:
x = []
y = []
z = []
vx = [1.0] * numCorpos
vy = [1.0] * numCorpos
vz = [1.0] * numCorpos
m = [1.0] * numCorpos
xTemp = 0
yTemp = 0
zTemp = 0

for i in range(corposPorEixo3D):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo3D):
    yTemp += espacamento
    zTemp = 0
    for k in range(corposPorEixo3D):
      zTemp += espacamento
      x.append(xTemp)
      y.append(yTemp)
      z.append(zTemp)

if (resto != 0):
    for i in range (resto):
        xTemp += espacamento
        x.append(xTemp)
        yTemp += espacamento
        y.append(yTemp)
        zTemp += espacamento
        z.append(zTemp)

Aqui jogamos os arrays criados com Python puro para o Numpy

In [ ]:
np_x = np.array(x, dtype=np.float64)
np_y = np.array(y, dtype=np.float64)
np_z = np.array(z, dtype=np.float64)
np_m = np.array(m, dtype=np.float64)
np_vx = np.array(vx, dtype=np.float64)
np_vy = np.array(vy, dtype=np.float64)
np_vz = np.array(vz, dtype=np.float64)

Função de cálculo da força gravitacional de cada partícula dentro do sistema. A função retorna a força x, y e z total para cada partícula

In [ ]:
def calcularForcas(np_x, np_y, np_z, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[:, None]
    dy = np_y[None, :] - np_y[:, None]
    dz = np_z[None, :] - np_z[:, None]

    dist = np.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (np_m[:, None] * np_m[None, :]) / (dist**2)

    fx = F * (dx / dist)
    fy = F * (dy / dist)
    fz = F * (dz / dist)

    fx_total = np.sum(fx, axis=1)
    fy_total = np.sum(fy, axis=1)
    fz_total = np.sum(fz, axis=1)

    return fx_total, fy_total, fz_total

Usando o método integrador Euler-Cromer criamos a função que ira simular a interação entre as partículas num intervalo de tempo atualizando as posições e velocidades x, y e z de cada partícula

In [ ]:
def atualizarPosicao(np_x, np_y, np_z, np_vx, np_vy, np_vz, np_m, dt, G, epsilon):
    fx, fy, fz = calcularForcas(np_x, np_y, np_z, np_m, G, epsilon)

    ax = fx / np_m
    ay = fy / np_m
    az = fz / np_m

    np_vx += ax * dt
    np_vy += ay * dt
    np_vz += az * dt

    np_x += np_vx * dt
    np_y += np_vy * dt
    np_z += np_vz * dt

Função que calcula a energia cinética total do sistema

In [ ]:
def calcularEnergiaCinetica(np_vx, np_vy, np_vz, np_m):
    return 0.5 * np.sum(np_m * (np_vx**2 + np_vy**2 + np_vz**2))

Função que calcula a energia potencial total do sistema

In [ ]:
def calcularEnergiaPotencial(np_x, np_y, np_z, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[: ,None]
    dy = np_y[None, :] - np_y[: ,None]
    dz = np_z[None, :] - np_z[: ,None]

    dist = np.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = np_m[:, None] * np_m[None, :]

    U = G * np.sum(m_prod / dist) / 2

    return U

---

Execução

In [ ]:
%%time
for i in range(passos):
    atualizarPosicao(np_x, np_y, np_z, np_vx, np_vy, np_vz, np_m, dt, G, epsilon)

CPU times: user 18.1 s, sys: 19.6 s, total: 37.8 s
Wall time: 39.8 s


---
## **Cupy**
---

### <font color="red">3D</font>

Inicialização dos atributos feita em Python puro, essa parte do código sera executada apenas uma vez. Os vetores gerados serão jogados para o Cupy depois

In [ ]:
x = []
y = []
z = []
vx = [1.0] * numCorpos
vy = [1.0] * numCorpos
vz = [1.0] * numCorpos
m = [1.0] * numCorpos
xTemp = 0
yTemp = 0
zTemp = 0

for i in range(corposPorEixo3D):
  xTemp += espacamento
  yTemp = 0
  for j in range(corposPorEixo3D):
    yTemp += espacamento
    zTemp = 0
    for k in range(corposPorEixo3D):
      zTemp += espacamento
      x.append(xTemp)
      y.append(yTemp)
      z.append(zTemp)

if (resto != 0):
    for i in range (resto):
        xTemp += espacamento
        x.append(xTemp)
        yTemp += espacamento
        y.append(yTemp)
        zTemp += espacamento
        z.append(zTemp)

Aqui jogamos os arrays criados com Python puro para o Cupy

In [ ]:
cp_x = cp.array(x, dtype=cp.float64)
cp_y = cp.array(y, dtype=cp.float64)
cp_z = cp.array(z, dtype=cp.float64)
cp_m = cp.array(m, dtype=cp.float64)
cp_vx = cp.array(vx, dtype=cp.float64)
cp_vy = cp.array(vy, dtype=cp.float64)
cp_vz = cp.array(vz, dtype=cp.float64)

Função de cálculo da força gravitacional de cada partícula dentro do sistema. A função retorna a força x, y e z total para cada partícula

In [ ]:
def calcularForcas(cp_x, cp_y, cp_z, cp_m, G, epsilon):
    dx = cp_x[None, :] - cp_x[:, None]
    dy = cp_y[None, :] - cp_y[:, None]
    dz = cp_z[None, :] - cp_z[:, None]

    dist = cp.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (cp_m[:, None] * cp_m[None, :]) / (dist**2)

    fx = F * (dx / dist)
    fy = F * (dy / dist)
    fz = F * (dz / dist)

    fx_total = cp.sum(fx, axis=1)
    fy_total = cp.sum(fy, axis=1)
    fz_total = cp.sum(fz, axis=1)

    return fx_total, fy_total, fz_total

Usando o método integrador Euler-Cromer criamos a função que ira simular a interação entre as partículas num intervalo de tempo atualizando as posições e velocidades x, y e z de cada partícula

In [ ]:
def atualizarPosicao(cp_x, cp_y, cp_z, cp_vx, cp_vy, cp_vz, cp_m, dt, G, epsilon):
    fx, fy, fz = calcularForcas(cp_x, cp_y, cp_z, cp_m, G, epsilon)

    ax = fx / cp_m
    ay = fy / cp_m
    az = fz / cp_m

    cp_vx += ax * dt
    cp_vy += ay * dt
    cp_vz += az * dt

    cp_x += cp_vx * dt
    cp_y += cp_vy * dt
    cp_z += cp_vz * dt

Função que calcula a energia cinética total do sistema

In [ ]:
def calcularEnergiaCinetica(cp_vx, cp_vy, cp_vz, cp_m):
    return 0.5 * cp.sum(cp_m * (cp_vx**2 + cp_vy**2 + cp_vz**2))

Função que calcula a energia potencial total do sistema

In [ ]:
def calcularEnergiaPotencial(cp_x, cp_y, cp_z, cp_m, G, epsilon):
    dx = cp_x[None, :] - cp_x[: ,None]
    dy = cp_y[None, :] - cp_y[: ,None]
    dz = cp_z[None, :] - cp_z[: ,None]

    dist = cp.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = cp_m[:, None] * cp_m[None, :]

    U = G * cp.sum(m_prod / dist) / 2

    return U

---

Execução

In [ ]:
%%time
for i in range(passos):
    atualizarPosicao(cp_x, cp_y, cp_z, cp_vx, cp_vy, cp_vz, cp_m, dt, G, epsilon)

CPU times: user 1.53 s, sys: 136 ms, total: 1.66 s
Wall time: 2.02 s
